# 04b · PyTorch lấy đạo hàm như thế nào?

**Chỉ học autograd, chưa train PINN.** Mục tiêu: hiểu `requires_grad=True`, `torch.autograd.grad`, `grad_outputs`, `create_graph=True`; kiểm tra đạo hàm bậc bốn bằng một đa thức biết đáp án. Mỗi đầu ra ở hàng $i$ chỉ phụ thuộc đầu vào cùng hàng.

## 1. Đạo hàm bậc một

PyTorch ghi lại các phép toán tạo $y=x^3$. Tọa độ phải có `requires_grad=True`. `grad_outputs=torch.ones_like(y)` có cùng shape với $y$ và cho PyTorch biết ta đang lấy tổng có trọng số 1 của các đầu ra; vì các hàng độc lập, kết quả là đạo hàm theo từng điểm.

In [ ]:
import torch

x = torch.tensor([[0.5], [1.0]], dtype=torch.float64, requires_grad=True)
y = x**3
dy = torch.autograd.grad(y, x, torch.ones_like(y), create_graph=True)[0]
print("autograd:", dy.reshape(-1).tolist())
print("đạo hàm tay 3*x**2:", (3*x**2).reshape(-1).tolist())

## 2. Vì sao `create_graph=True`?

Ta cần đạo hàm của **chính đạo hàm vừa tính**. Với $p(x)=x^4-2x^3+x$, kết quả mong đợi là $p^{(4)}(x)=24$. Viết từng bước để nhìn rõ bậc 1 → 2 → 3 → 4.

In [ ]:
x = torch.tensor([[0.25], [0.75]], dtype=torch.float64, requires_grad=True)
p = x**4 - 2*x**3 + x
p1 = torch.autograd.grad(p, x, torch.ones_like(p), create_graph=True)[0]
p2 = torch.autograd.grad(p1, x, torch.ones_like(p1), create_graph=True)[0]
p3 = torch.autograd.grad(p2, x, torch.ones_like(p2), create_graph=True)[0]
p4 = torch.autograd.grad(p3, x, torch.ones_like(p3), create_graph=True)[0]
print("p' tại hai điểm:", p1.reshape(-1).tolist())
print("p'''' tại hai điểm:", p4.reshape(-1).tolist())
assert torch.allclose(p4, torch.full_like(p4, 24.0))

## 3. Một hàm nhỏ để khỏi chép cùng câu lệnh

Bài PINN sẽ gọi autograd sáu lần trong mỗi bước train. Ta gói **một lệnh** vào hàm `grad(y,x)`; bản thân hàm không có phép toán mới.

In [ ]:
def grad(y, x):
    return torch.autograd.grad(y, x, torch.ones_like(y), create_graph=True)[0]

q1 = grad(p, x)
q2 = grad(q1, x)
q3 = grad(q2, x)
q4 = grad(q3, x)
print("hàm grad cho p'''' =", q4.reshape(-1).tolist())

## 4. Tách hai loại gradient

`grad(u,x)` là đạo hàm theo **tọa độ**, để lập residual phương trình. Sau này `loss.backward()` tính đạo hàm của loss theo **trọng số mạng**; `optimizer.step()` mới cập nhật trọng số. `grad` không tự train mạng.

**Tự làm:** trong một ô bản sao, bỏ `requires_grad=True` và ghi lỗi. Khôi phục ô gốc rồi thay $p(x)$ bằng $x^4$; đạo hàm bậc bốn thay đổi không? **Sản phẩm:** giá trị bậc một và bậc bốn kèm giải thích `create_graph=True`.

**Tiếp theo:** [04c · pipeline PINN](04c-pinn-pipeline.ipynb).